# 08 — Blending dos Modelos
## PRT Seguros

Ideia simples e barata: em vez de escolher **um** modelo, combinamos as probabilidades de vários.
Isso costuma reduzir variância — cada modelo comete erros diferentes (a Regressão Logística é linear,
o Random Forest é bagging, os 3 de boosting são sequenciais mas com estratégias de árvore diferentes),
e a média tende a cancelar parte desses erros individuais.

Testamos algumas combinações no conjunto de **validação** (o mesmo split para os 5 modelos, salvo
por cada notebook em `dados_processados/proba_val/`) e escolhemos a que der o maior AUC-ROC antes
de gerar a submissão final para o Kaggle.

## 1. Carregar probabilidades de validação de cada modelo

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

val = pd.read_csv("dados_processados/val_model_ready.csv")[["cod_individuo", "churned"]]

MODELOS = ["logistic_regression", "random_forest", "xgboost", "lightgbm", "catboost"]

proba_val = val[["cod_individuo"]].copy()
for m in MODELOS:
    p = pd.read_csv(f"dados_processados/proba_val/{m}.csv")
    proba_val = proba_val.merge(p.rename(columns={"proba": m}), on="cod_individuo", how="left")

proba_val = proba_val.merge(val, on="cod_individuo", how="left")
assert proba_val[MODELOS].isna().sum().sum() == 0
proba_val.head()


,cod_individuo,logistic_regression,random_forest,xgboost,lightgbm,catboost,churned
0,221300304747,0.553416,0.468449,0.488496,0.213170,0.515937,0
1,221301534791,0.871557,0.811365,0.906195,0.290202,0.921939,0
2,221303166616,0.087915,0.097729,0.121188,0.116002,0.084966,0
3,221302578576,0.200263,0.116751,0.141469,0.122275,0.145739,0
4,221302165563,0.662075,0.601387,0.705160,0.252214,0.696310,1


## 2. AUC individual de cada modelo (conferência)

In [2]:
aucs_individuais = {
    m: roc_auc_score(proba_val["churned"], proba_val[m]) for m in MODELOS
}
for m, auc in sorted(aucs_individuais.items(), key=lambda x: -x[1]):
    print(f"{m:<22} AUC-ROC = {auc:.4f}")


catboost               AUC-ROC = 0.8254
xgboost                AUC-ROC = 0.8240
lightgbm               AUC-ROC = 0.8238
random_forest          AUC-ROC = 0.8196
logistic_regression    AUC-ROC = 0.8033


## 3. Testar estratégias de blending

- **Média simples dos 5**: todos os modelos pesam igual
- **Média simples dos 3 boosting** (XGBoost + LightGBM + CatBoost): remove os dois modelos mais fracos
- **Média ponderada pelo AUC**: cada modelo pesa proporcional ao quanto (AUC − 0.5) ele contribui —
  modelo melhor pesa mais, mas nenhum é zerado

In [3]:
resultados_blend = {}

# Média simples de todos os 5
proba_blend_5 = proba_val[MODELOS].mean(axis=1)
resultados_blend["media_5_modelos"] = roc_auc_score(proba_val["churned"], proba_blend_5)

# Média simples dos 3 boosting
BOOSTING = ["xgboost", "lightgbm", "catboost"]
proba_blend_boosting = proba_val[BOOSTING].mean(axis=1)
resultados_blend["media_3_boosting"] = roc_auc_score(proba_val["churned"], proba_blend_boosting)

# Média ponderada pelo (AUC - 0.5) de cada modelo
pesos = {m: max(aucs_individuais[m] - 0.5, 0) for m in MODELOS}
soma_pesos = sum(pesos.values())
pesos = {m: w / soma_pesos for m, w in pesos.items()}
proba_blend_ponderada = sum(proba_val[m] * pesos[m] for m in MODELOS)
resultados_blend["media_ponderada_5"] = roc_auc_score(proba_val["churned"], proba_blend_ponderada)

print("Pesos da média ponderada:")
for m, w in sorted(pesos.items(), key=lambda x: -x[1]):
    print(f"  {m:<22} peso={w:.3f}")

print()
print("=== AUC-ROC de cada estratégia de blending ===")
for nome, auc in sorted(resultados_blend.items(), key=lambda x: -x[1]):
    print(f"{nome:<22} AUC-ROC = {auc:.4f}")

print()
melhor_individual = max(aucs_individuais, key=aucs_individuais.get)
print(f"Melhor modelo individual : {melhor_individual} ({aucs_individuais[melhor_individual]:.4f})")
melhor_blend = max(resultados_blend, key=resultados_blend.get)
print(f"Melhor blend             : {melhor_blend} ({resultados_blend[melhor_blend]:.4f})")


Pesos da média ponderada:
  catboost               peso=0.204
  xgboost                peso=0.203
  lightgbm               peso=0.203
  random_forest          peso=0.200
  logistic_regression    peso=0.190

=== AUC-ROC de cada estratégia de blending ===
media_3_boosting       AUC-ROC = 0.8255
media_ponderada_5      AUC-ROC = 0.8233
media_5_modelos        AUC-ROC = 0.8232

Melhor modelo individual : catboost (0.8254)
Melhor blend             : media_3_boosting (0.8255)


## 4. Gerar a submissão do melhor blend

Usamos as probabilidades **já calculadas** para o Kaggle em `submissions/submission_<modelo>.csv`
(nenhum modelo precisa ser retreinado) e aplicamos a mesma combinação que ganhou na validação.

In [4]:
submissoes = {}
for m in MODELOS:
    s = pd.read_csv(f"submissions/submission_{m}.csv")
    submissoes[m] = s.set_index("Id")["probabilidade_churn"]

kaggle_df = pd.DataFrame(submissoes)

if melhor_blend == "media_5_modelos":
    proba_final = kaggle_df[MODELOS].mean(axis=1)
elif melhor_blend == "media_3_boosting":
    proba_final = kaggle_df[BOOSTING].mean(axis=1)
else:  # media_ponderada_5
    proba_final = sum(kaggle_df[m] * pesos[m] for m in MODELOS)

submissao_blend = pd.DataFrame({
    "Id": kaggle_df.index,
    "probabilidade_churn": proba_final.values,
})
submissao_blend.to_csv("submissions/submission_blend.csv", index=False)

print(f"Estratégia usada: {melhor_blend} (AUC-ROC validação = {resultados_blend[melhor_blend]:.4f})")
print(f"Arquivo salvo: submissions/submission_blend.csv")
submissao_blend.head()


Estratégia usada: media_3_boosting (AUC-ROC validação = 0.8255)
Arquivo salvo: submissions/submission_blend.csv


,Id,probabilidade_churn
0,221300000002,0.117125
1,221300000020,0.256339
2,221300000097,0.136052
3,221300000148,0.334652
4,221300000166,0.255110
